In [1]:
import os

import numpy as np
import polars as pl
import torch
import sys

from modeling_module.models import build_patchTST_quantile
from modeling_module.utils.eval_utils import eval_on_loader_point, eval_on_loader_quantile

SRC = "/Users/igwanhyeong/PycharmProjects/ts_forecaster_lib/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)


# LTB modules (expected to exist in your repo)

# optional

'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = "/Users/igwanhyeong/PycharmProjects/ts_forecaster_lib/raw_data/"
WINDOW_DIR = "C:/Users/USER/PycharmProjects/ts_forecaster_lib/raw_data/"

DIR = WINDOW_DIR if sys.platform == "win32" else MAC_DIR
device = "cuda" if torch.cuda.is_available() else "cpu"

print("DIR:", DIR)
print("device:", device)
if device == "cuda":
    print("cuda:", torch.version.cuda, "gpu_count:", torch.cuda.device_count())

DIR: C:/Users/USER/PycharmProjects/ts_forecaster_lib/raw_data/
device: cuda
cuda: 12.8 gpu_count: 1


In [2]:
df = (pl.read_parquet(DIR + 'train_data/tb_master_target.parquet')
        .with_columns(
            pl.when(pl.col('demand_qty') == 0.0).then(1e-4).otherwise(pl.col('demand_qty')).alias('demand_qty')
        )
      )
df

oper_part_no,demand_dt,demand_qty,seq
str,i64,f64,u32
"""0001-1001""",201811,5.0,1
"""0001-1001""",201812,0.0001,2
"""0001-1001""",201813,0.0001,3
"""0001-1001""",201814,0.0001,4
"""0001-1001""",201815,0.0001,5
…,…,…,…
"""ZZ90239""",202622,0.0001,314
"""ZZ90239""",202623,0.0001,315
"""ZZ90239""",202624,0.0001,316


In [3]:
past_exo_cont_cols = (
    # "exo_p_y_lag_1w",
    # "exo_p_y_lag_2w",
    # "exo_p_y_lag_52w",
    # "exo_p_y_rollmean_4w","exo_p_y_rollmean_12w","exo_p_y_rollstd_4w",
    # "exo_p_weeks_since_holiday`",
    # "exo_p_temperature",
    # "exo_p_fuel_price",
    # "exo_p_cpi",
    # "exo_p_unemployment",
    # "exo_p_markdown_sum",
    # "exo_p_markdown1",
    # "exo_p_markdown2",
    # "exo_p_markdown3",
    # "exo_p_markdown4",
    # "exo_p_markdown5",
    # "exo_markdown1_isnull",
    # "exo_markdown2_isnull",
    # "exo_markdown3_isnull",
    # "exo_markdown4_isnull",
    # "exo_markdown5_isnull",
)
past_exo_cat_cols = (
    # "exo_c_woy_bucket",
)

lookback = 52
horizon = 27
batch_size = 512

freq = "weekly"          # walmart dt is weekly
split_mode = "multi"     # id-disjoint split (leakage-safe)
shuffle = True

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")

print("device:", device)

device: cuda


C:\Users\USER\python\py312\Lib\site-packages\torch\__init__.py:1612: UserWarning: This API is going to be deprecated, please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:81.)
  _C._set_float32_matmul_precision(precision)


In [4]:
# from modeling_module.utils.exogenous_utils import compose_exo_calendar_cb
#
# future_exo_cb_time = compose_exo_calendar_cb(date_type=freq)
#
# # ============================================================
# # Holiday lookup (vectorized) + FutureExo callback (time + holiday)
# # ============================================================
# holiday_map_dayidx = {
#     int(row[0]): float(row[1])
#     for row in (
#         df.select(["date_idx", "exo_is_holiday"])
#           .group_by("date_idx")
#           .agg(pl.max("exo_is_holiday").alias("exo_is_holiday"))
#           .sort("date_idx")
#           .iter_rows()
#     )
# }
#
# def build_holiday_array(holiday_map_dayidx: dict[int, float], *, pad: int = 0) -> np.ndarray:
#     if not holiday_map_dayidx:
#         return np.zeros((1,), dtype=np.float32)
#     max_k = max(int(k) for k in holiday_map_dayidx.keys())
#     arr = np.zeros((max_k + 1 + int(pad),), dtype=np.float32)
#     for k, v in holiday_map_dayidx.items():
#         kk = int(k)
#         if kk >= 0:
#             arr[kk] = float(v)
#     return arr
#
# class FutureExoTimePlusHoliday:
#     def __init__(self, holiday_by_dayidx: np.ndarray, *, step_days: int = 7):
#         self.holiday = holiday_by_dayidx.astype(np.float32, copy=False)
#         self.step_days = int(step_days)
#
#     def __call__(self, start_idx, H: int, device: str = "cpu"):
#         # 1) calendar exo (batch-safe)
#         cal = future_exo_cb_time(start_idx, H, device=device)  # scalar: (H,E) | batch: (B,H,E)
#
#         # 2) holiday exo (vectorized in numpy)
#         is_scalar = isinstance(start_idx, (int, np.integer))
#         if is_scalar:
#             s = np.asarray([int(start_idx)], dtype=np.int64)
#         else:
#             s = np.asarray(start_idx, dtype=np.int64).reshape(-1)
#
#         B = s.shape[0]
#         H = int(H)
#
#         offsets = (self.step_days * np.arange(H, dtype=np.int64))[None, :]  # (1,H)
#         idx = s[:, None] + offsets                                          # (B,H)
#
#         hol = np.zeros((B, H), dtype=np.float32)
#         valid = (idx >= 0) & (idx < self.holiday.shape[0])
#         hol[valid] = self.holiday[idx[valid]]
#         hol_t = torch.from_numpy(hol).unsqueeze(-1)                         # (B,H,1), CPU
#
#         target_device = cal.device  # cal이 이미 cuda일 수 있음
#         cal = cal.to(target_device, dtype=torch.float32)
#         hol_t = hol_t.to(target_device, dtype=torch.float32)
#
#         # 3) concat
#         if is_scalar:
#             out = torch.cat([cal.to(torch.float32).unsqueeze(0), hol_t], dim=-1)[0]  # (H,E+1)
#         else:
#             out = torch.cat([cal.to(torch.float32), hol_t], dim=-1)                  # (B,H,E+1)
#
#         return out
#
# holiday_by_dayidx = build_holiday_array(holiday_map_dayidx, pad=7 * (horizon + 2))
# future_exo_cb_time_plus_holiday = FutureExoTimePlusHoliday(holiday_by_dayidx, step_days=7)


In [5]:
from torch.utils.tensorboard import SummaryWriter
import os

tb_logdir = r"C:\Users\USER\PycharmProjects\ts_forecaster_lib\src\model_test\tb_prof"
os.makedirs(tb_logdir, exist_ok=True)

w = SummaryWriter(tb_logdir)
w.add_scalar("dummy/alive", 1.0, 0)
w.flush()
w.close()

In [6]:
from modeling_module.data_loader.multi_part_exo_data_module import MultiPartExoDataModule
from modeling_module.models import build_patchTST
from modeling_module.training.model_losses.loss_module import DistributionLoss, MQLoss, HuberLoss, Pinball, \
    MultiQuantilePinball, MSE
from modeling_module.utils.metrics import mae, rmse, smape
from modeling_module.utils.checkpoint import load_model_dict

from modeling_module.training.model_trainers.total_train import run_total_train_weekly
import pandas as pd
import random

rows = []  # seed loop 밖에서 선언


def set_seed(seed: int = 11):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    elif torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_datamodule(variant: str) -> MultiPartExoDataModule:
    if variant == "A0":
        future_exo_cb = None
    # elif variant == "A1":
    #     future_exo_cb = future_exo_cb_time
    # elif variant == "A2":
    #     future_exo_cb = future_exo_cb_time_plus_holiday
    else:
        raise ValueError(variant)

    return MultiPartExoDataModule(
        df=df,
        id_col="oper_part_no",
        date_col="demand_dt",
        y_col="demand_qty",
        lookback=lookback,
        horizon=horizon,
        batch_size=batch_size,
        past_exo_cont_cols=past_exo_cont_cols,
        past_exo_cat_cols=past_exo_cat_cols,
        future_exo_cb=future_exo_cb,
        freq=freq,
        shuffle=shuffle,
        split_mode=split_mode,
    )


def inspect(loader, name):
    b = next(iter(loader))
    x, y, uid, fe, pe_cont, pe_cat = b
    print(f"[{name}] x:", x.shape, x.device, x.dtype)
    print(f"[{name}] fe:", fe.shape, fe.device, fe.dtype)
    print(f"[{name}] pe:", pe_cont.shape, pe_cont.device, pe_cont.dtype)
    print(f"[{name}] future_exo_cb is None?", loader.collate_fn.future_exo_cb is None)
    if fe.shape[-1] > 0:
        print(f"[{name}] fe sample:", fe[0, :3, :])


def to_point_pred(y_hat, *, prefer="loc", param_names=None):
    """
    y_hat:
      - point: (B,H) or (B,H,1)
      - dist packed: (B,H,M)
      - dict output: {"y_hat": ...} or {"loc": ...}
    return:
      - (B,H) point prediction
    """
    # dict 형태 대응
    if isinstance(y_hat, dict):
        if "y_hat" in y_hat:
            y_hat = y_hat["y_hat"]
        elif prefer in y_hat:           # e.g. "loc"
            return np.asarray(y_hat[prefer])
        elif "loc" in y_hat:
            return np.asarray(y_hat["loc"])

    y_hat = np.asarray(y_hat)

    # (B,H) already point
    if y_hat.ndim == 2:
        return y_hat

    # (B,H,1) -> squeeze
    if y_hat.ndim == 3 and y_hat.shape[-1] == 1:
        return y_hat[..., 0]

    # (B,H,M) distribution packed -> pick loc
    if y_hat.ndim == 3 and y_hat.shape[-1] >= 2:
        M = y_hat.shape[-1]

        # param_names 있으면 loc index 우선
        if param_names is not None:
            try:
                loc_idx = list(param_names).index("loc")
                return y_hat[..., loc_idx]
            except Exception:
                pass

        # fallback convention
        # Normal:   [loc, scale] => loc=0
        # StudentT: [df, loc, scale] => loc=1
        if M == 3:
            return y_hat[..., 1]
        return y_hat[..., 0]

    raise ValueError(f"Unsupported y_hat shape: {y_hat.shape}")


def squeeze_y(y):
    y = np.asarray(y)
    if y.ndim == 3 and y.shape[-1] == 1:
        return y[..., 0]
    return y
save_dir = os.path.join(DIR, "fit", "walmart_patchtst_ab")
os.makedirs(save_dir, exist_ok=True)

save_root_A0 = os.path.join(save_dir, "DSIO_RUNNING", )

data_module_A0 = build_datamodule("A0")

train_loader_A0 = data_module_A0.get_train_loader(batch_size=batch_size, shuffle=True, num_workers=0)
val_loader_A0 = data_module_A0.get_val_loader()

inspect(train_loader_A0, "A0")

# ============================================
# 학습 실행 (LTB total_train 포맷 유지)
# - 여기서는 "외생변수 A/B/C"만 비교하므로 use_ssl_pretrain=False로 고정
# Walmart처럼 항상 판매량이 있는(continuous) 데이터에서 “스파이크”를 잡는 규칙이:
# 스파이크 마스크가 과도하게 넓게 잡히거나(사실상 대부분 True)
# spike-loss가 MSE/제곱오차 기반인데 reduction이 sum 또는 정규화 없이 누적되어
# sales 스케일(1e4~1e5)에서 제곱오차가 1e8급으로 바로 올라가
# → 결과적으로 delta가 1e8 수준으로 튄다.
# → 그래서 최종적으로 이 상황에서는 spike_epoch를 0으로 잡아준다.
# ============================================
print('run result_A0')
results_A0 = run_total_train_weekly(
    train_loader_A0,
    val_loader_A0,
    device=device,
    lookback=lookback,
    horizon=horizon,
    warmup_epochs=2,
    spike_epochs=1,
    ssl_pretrain_epochs=3,
    # warmup_epochs=1,
    # spike_epochs=1,
    save_dir=save_root_A0,
    # loss = DistributionLoss(distribution="StudentT", level=[80, 90]),
    loss = HuberLoss(delta = 5.0),
    # loss=MSE(),
    # loss_quantile=MQLoss(quantiles=[0.1,0.5,0.99]),  # Quantile 학습용
    loss_quantile=MultiQuantilePinball(),
    use_ssl_mode='full',
    use_exogenous_mode=False,
    models_to_run=[
                    "patchtst",
                   "patchmixer",
                   'titan'
    ],
)

builders = {
    'PatchTSTQuantile': build_patchTST_quantile,
    'PatchTST': build_patchTST,
}

# print(load_model_dict(save_root_A0, builders, device=device))
#
model_A0 = load_model_dict(save_root_A0, builders, device=device)['patchtst']

y0, yhat0 = eval_on_loader_point(
    model_A0,
    val_loader_A0,
    device = device,
    future_exo_cb=None,
    use_exo_inputs=False,
    mismatch_policy="error",
    lookback=52,
    horizon=26,
)
pn0 = getattr(getattr(model_A0, "cfg", None), "param_names", None)
yhat0_point = to_point_pred(yhat0, param_names=pn0)

metric_A0 = {
    "MAE": float(mae(y0.reshape(-1), yhat0_point.reshape(-1))),
    "RMSE": float(rmse(y0.reshape(-1), yhat0_point.reshape(-1))),
    "SMAPE": float(smape(y0.reshape(-1), yhat0_point.reshape(-1))),
}

model_A0_q = load_model_dict(save_root_A0, builders, device=device)["patchtst_quantile"]

y0_q, yhat0_q = eval_on_loader_quantile(
    model_A0_q, val_loader_A0, device,
    use_exo_inputs=False,
    lookback=52,
    horizon=26,
)


metric_A0_q = {
    "MAE": float(mae(y0_q.reshape(-1), yhat0_q.reshape(-1))),
    "RMSE": float(rmse(y0_q.reshape(-1), yhat0_q.reshape(-1))),
    "SMAPE": float(smape(y0_q.reshape(-1), yhat0_q.reshape(-1))),
}

# -------------------------
# Point metrics row append
# -------------------------
rows.append({
    "variant": "A0",
    "model_type": "point",
    "MAE": metric_A0["MAE"],
    "RMSE": metric_A0["RMSE"],
    "SMAPE": metric_A0["SMAPE"],
    "save_root": save_root_A0,
})


# -------------------------
# Quantile metrics row append
# (주의: q50 등 기준이 명확해야 함)
# -------------------------
rows.append({
    "variant": "A0",
    "model_type": "quantile(q50)",
    "MAE": metric_A0_q["MAE"],
    "RMSE": metric_A0_q["RMSE"],
    "SMAPE": metric_A0_q["SMAPE"],
    "save_root": save_root_A0,
})


# loop 종료 후 저장
df_out = pd.DataFrame(rows)

out_csv = os.path.join(save_dir, "ab_results_by_seed.csv")
df_out.to_csv(out_csv, index=False)

# variant별 mean/std 요약도 같이 저장 추천
summary = (
    df_out.groupby(["variant", "model_type"])[["MAE", "RMSE", "SMAPE"]]
    .agg(["mean", "std"])
    .reset_index()
)
out_sum = os.path.join(save_dir, "ab_results_summary.csv")
summary.to_csv(out_sum, index=False)

# print(metric_A0)
# print(metric_A0_q)

print("saved:", out_csv, out_sum)

[A0] x: torch.Size([512, 52, 1]) cpu torch.float32
[A0] fe: torch.Size([512, 27, 0]) cpu torch.float32
[A0] pe: torch.Size([512, 52, 0]) cpu torch.float32
[A0] future_exo_cb is None? True
run result_A0
[exo_policy] use_exo=False | future(has=True, dim=0) | past(cont=0, cat=0)
[total_train][EXO] use_exo=False source=none exo_dim=0 future_cb=False past_cont=0 past_cat=0

[total_train] === RUN: patchtst (weekly) ===
[SSL] PatchTST Pretrain (Weekly) -> C:\Users\USER\PycharmProjects\ts_forecaster_lib\raw_data\fit\walmart_patchtst_ab\DSIO_RUNNING\pretrain\patchtst_pretrain_best.pt
[train_patchtst_pretrain] Effective TrainingConfig:
{
  "device": "cuda",
  "log_every": 100,
  "lookback": 52,
  "horizon": 27,
  "epochs": 0,
  "lr": 0.0001,
  "weight_decay": 0.001,
  "t_max": 40,
  "patience": 100,
  "max_grad_norm": 30.0,
  "amp_device": "cuda",
  "use_amp": true,
  "loss_mode": "auto",
  "point_loss": "mse",
  "huber_delta": 0.8,
  "q_star": 0.5,
  "use_cost_q_star": false,
  "Cu": 2.0,
  "Co